# Waypoint-6m fine-tuning webinar — regression

A minimal walkthrough of `waypoint finetune` on **Compass task 6** (`mastrorilli`): fine-tune the pretrained model on drug degradation rate and read the head's held-out test metrics.

Every heavy step is a `waypoint` CLI call and its output caches to `artifacts/task6_degradation/`. The default is `USE_CACHE = True` — on the first run each step computes and caches; on subsequent runs the cache is loaded and the whole notebook finishes in seconds. Set `USE_CACHE = False` to force a rerun.

The parallel classification demo is in [`webinar_finetune_classification.ipynb`](webinar_finetune_classification.ipynb). Both notebooks share [`finetune.yaml`](finetune.yaml) and [`webinar_utils.py`](webinar_utils.py).

## Setup

Imports (from the shared [`webinar_utils.py`](webinar_utils.py) module), tunable knobs (`USE_CACHE`, `TASK` dict with target/covariate), and the artifact directory setup.

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datasets import load_dataset

from webinar_utils import (
    BASE_MODEL_ID, COMPASS_REPO,
    paths_for,
)

# --- knobs -----------------------------------------------------------------
# USE_CACHE=True   →  each step loads its cached artifact if present, else computes and caches.
# USE_CACHE=False  →  each step recomputes from scratch and overwrites the cache.
# Default is True so that after the first (slow) run, the webinar plays back instantly.
USE_CACHE = True

WEBINAR_DIR = Path.cwd()
FINETUNE_CONFIG = WEBINAR_DIR / "finetune.yaml"
ARTIFACT_DIR = WEBINAR_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

# Task config for Compass task 6 (regression on drug degradation rate).
TASK = {
    "hub_config": "mastrorilli",
    "target": "Degradation Rate",
    "covariate": "Drug",
    "task_type": "regression",
    "tag": "task6_degradation",
    "config": FINETUNE_CONFIG,
}

PATHS = paths_for(ARTIFACT_DIR, TASK["tag"])

# Persisted between cells — accumulates wall-clock timings and money-slide metrics.
timings: dict[str, float] = (
    json.loads(PATHS["timings"].read_text()) if USE_CACHE and PATHS["timings"].exists() else {}
)
metrics: dict[str, dict] = (
    json.loads(PATHS["metrics"].read_text()) if USE_CACHE and PATHS["metrics"].exists() else {}
)
print(f"USE_CACHE={USE_CACHE}   |   waypoint CLI auto-detects device (cuda/mps/cpu)")
print(f"Webinar dir: {WEBINAR_DIR}")
print(f"Artifact dir for this task: {PATHS['samples'].parent}")
print(f"Config: {FINETUNE_CONFIG.name}  {'✓' if FINETUNE_CONFIG.exists() else '(missing!)'}")

## 1. Load Compass task 6 (`mastrorilli`)

The `mastrorilli` config is a drug-degradation screen: each sample is a stool community incubated with one of 265 drugs, and the target is how fast that drug is metabolized.

Every row is one sample in the **waypoint format** — the shape every `waypoint` CLI command expects:

| column | type | purpose |
|---|---|---|
| `Taxa` | `list[str]` | taxonomic labels present in the sample |
| `Relative Abundances` | `list[float]` | matching proportions, sum to ~1 |
| `Degradation Rate` | `float` | regression target |
| `Drug` | `str` | covariate — which drug was applied |

The cell below writes `artifacts/task6_degradation/samples.parquet` — the `--data` input for every embed / fine-tune call. To run on your own data, produce a parquet in this shape (or run `waypoint prepare-dataset` on an abundance matrix).

In [ ]:
if USE_CACHE and PATHS["samples"].exists():
    df = pd.read_parquet(PATHS["samples"])
    print(f"Loaded cached samples from {PATHS['samples']}")
else:
    # Pull the mastrorilli config from Compass and pool all splits.
    ds = load_dataset(COMPASS_REPO, TASK["hub_config"])
    frames = [ds[s].to_pandas() for s in ("train", "validation", "test") if s in ds]
    df = pd.concat(frames, ignore_index=True)
    # Drop rows where the target is missing or non-numeric.
    df[TASK["target"]] = pd.to_numeric(df[TASK["target"]], errors="coerce")
    df = df.dropna(subset=[TASK["target"]]).reset_index(drop=True)
    # Persist in waypoint format — the --data file every CLI call downstream reads.
    df[["Taxa", "Relative Abundances", TASK["target"], TASK["covariate"]]].to_parquet(PATHS["samples"])
    print(f"Downloaded and cached {len(df):,} samples to {PATHS['samples']}")

print(f"{len(df):,} samples   |   {df[TASK['covariate']].nunique()} unique drugs")
df[[TASK['covariate'], TASK['target']]].head(5)

In [ ]:
# Sanity check — sample count, unique drug count, and how many target values were NaN.
# Catches surprises like "most rows got filtered out by the numeric coerce".
print("df rows:", len(df))
print("unique drugs:", df[TASK["covariate"]].nunique())
print(df[TASK["covariate"]].value_counts())
print("nulls in target:", df[TASK["target"]].isna().sum())

In [ ]:
# Target distribution overlaid per drug — spread within each drug tells you how
# much per-drug variance a model has to explain.
px.histogram(
    df, x=TASK["target"], color=TASK["covariate"], nbins=40, opacity=0.75,
    title=f"Target distribution: {TASK['target']} by {TASK['covariate']}",
).update_layout(barmode="overlay")

## 2. Fine-tune the model

Runs `waypoint finetune` in a subshell using the shared [`finetune.yaml`](finetune.yaml) config. Output goes to `artifacts/task6_degradation/finetune_run/` — including the checkpoint (`best_model/`) and the held-out `test_metrics.json` we read in section 3.

```bash
waypoint finetune \
    --model outpost-bio/Waypoint-6m \
    --data artifacts/task6_degradation/samples.parquet \
    --output_dir artifacts/task6_degradation/finetune_run \
    --task_type regression \
    --target "Degradation Rate" \
    --covariate_column Drug \
    --config finetune.yaml
```

With `USE_CACHE=True` (default), if the checkpoint already exists on disk from a previous run, this cell just prints the path and moves on — no retraining.

In [ ]:
if USE_CACHE and PATHS["ft_model"].exists():
    print(f"Using cached fine-tuned checkpoint at {PATHS['ft_model']}")
else:
    PATHS["ft_dir"].mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    !waypoint finetune \
        --model {BASE_MODEL_ID} \
        --data "{PATHS['samples']}" \
        --output_dir "{PATHS['ft_dir']}" \
        --task_type {TASK['task_type']} \
        --target "{TASK['target']}" \
        --covariate_column {TASK['covariate']} \
        --config "{TASK['config']}"
    key = f"{TASK['tag']}__finetune_s"
    timings[key] = round(time.time() - t0, 2)
    print(f"Fine-tune finished in {timings[key]}s")

if not PATHS["ft_model"].exists():
    raise FileNotFoundError(f"Fine-tune did not produce {PATHS['ft_model']}")

### Training + eval loss curve

`waypoint finetune` writes `training_log.csv` (every logged step's train + eval loss) and a matching `training_log.html` into the output dir. The cell below reads the CSV and renders the plot inline.

In [ ]:
log_path = PATHS["ft_dir"] / "training_log.csv"
log_df = pd.read_csv(log_path)
print(f"Loaded {len(log_df):,} log rows from {log_path}")

fig = go.Figure()
if "loss" in log_df.columns:
    train = log_df.dropna(subset=["loss"])
    fig.add_scatter(x=train["step"], y=train["loss"], mode="lines", name="train loss")
if "eval_loss" in log_df.columns:
    eval_ = log_df.dropna(subset=["eval_loss"])
    fig.add_scatter(x=eval_["step"], y=eval_["eval_loss"], mode="lines+markers", name="eval loss")
fig.update_layout(title="Fine-tune loss over training steps",
                  xaxis_title="step", yaxis_title="loss",
                  legend_title="", template="plotly_white")
fig

## 3. Held-out test metrics

`waypoint finetune` writes `test_metrics.json` (and `validation_metrics.json`) into the output directory alongside `best_model/`. The `score` field is the benchmark-equivalent number for this task — R² for regression.

In [ ]:
test_metrics_path = PATHS["ft_dir"] / "test_metrics.json"
if not test_metrics_path.exists():
    raise FileNotFoundError(
        f"Missing {test_metrics_path}. The fine-tune step should have written it — "
        f"rerun section 2 with USE_CACHE=False if it's absent."
    )
test_metrics = json.loads(test_metrics_path.read_text())
print("=== Held-out TEST metrics (fine-tuned head, benchmark-equivalent) ===")
print(json.dumps(test_metrics, indent=2))

val_metrics_path = PATHS["ft_dir"] / "validation_metrics.json"
if val_metrics_path.exists():
    print("\n=== Validation metrics ===")
    print(json.dumps(json.loads(val_metrics_path.read_text()), indent=2))

# Stash into the notebook's metrics dict for persistence.
metrics[TASK["tag"]] = {
    "target": TASK["target"],
    "n_samples": int(len(df)),
    "test_score":   test_metrics.get("score"),
    "test_metrics": test_metrics.get("metrics"),
}

## Persist metrics + timings

Writes `metrics.json` (money-slide numbers) and `timings.json` (wall-clock per step) into this task's `artifacts/` subdirectory. Handy for pulling into a slide deck or comparing runs later.

In [ ]:
PATHS["metrics"].write_text(json.dumps(metrics, indent=2))
PATHS["timings"].write_text(json.dumps(timings, indent=2))
print(f"Wrote {PATHS['metrics']}")
print(f"Wrote {PATHS['timings']}")
print(json.dumps({"metrics": metrics, "timings": timings}, indent=2))

## Bring your own data

Two shapes of parquet input, no other changes required.

**Already have taxa + abundance lists?** Save a parquet in waypoint format:

```python
df.to_parquet("my_data.parquet")   # columns: Taxa (list[str]), Relative Abundances (list[float]), <target>, [covariate]
```

**Have a raw abundance matrix (taxa × samples TSV + sample metadata CSV)?** Convert first:

```bash
waypoint prepare-dataset --input matrix.tsv --output my_data.parquet --metadata labels.csv
```

Then fine-tune and read the head's test metrics. Copy `finetune.yaml` next to your data and tweak `num_epochs` / `learning_rate` / `warmup_steps` for your dataset size:

```bash
waypoint finetune --model outpost-bio/Waypoint-6m --data my_data.parquet \
                  --output_dir outputs/my_finetune \
                  --task_type regression --target "My Target" \
                  --covariate_column MyCovariate \
                  --config finetune.yaml
# Read outputs/my_finetune/test_metrics.json for the benchmark-equivalent number.
```

Flip `use_lora: true` in the YAML if you need parameter-efficient tuning to fit a smaller GPU.